<a href="https://colab.research.google.com/github/MayerT1/AGU_Session_Hunter/blob/main/AGU_Presentation_Title_Abstract_Hunter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AGU Sessions or Presentation Titles and Abstracts Matching Multiple Keywords

Customize keywords under configuration, and optionally True/False for additional customized expanded search words.

## Presentation Search

Keyword match in presentation titles and abstracts (not sessions). Takes as long as 30 min to run if including many keywords.

In [ ]:
# ============================================================
#   Search EVERY AGU26 presentation (title and abstract) for your keywords.
#
#   The meeting app (studio.m-anage.com) loads its program from a JSON API:
#       Program -> Session -> Paper   (each Paper record contains the full "Abstract")
#   1. Open the app once in headless Chromium (so requests are same-origin) and
#      crawl Home -> Program -> Session -> Paper through that API.
#   2. Cache every paper's title/abstract/metadata to disk (re-runs with new keywords are instant).
#   3. Match your keywords in the title and in the abstract, write one csv.
# ============================================================
!pip install -q pandas playwright nest_asyncio beautifulsoup4
!playwright install --with-deps chromium

import asyncio
import json
import os
import re
import time

import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

nest_asyncio.apply()  # lets asyncio.run() work inside Colab/Jupyter

# ---- Configuration ----
APP_URL = "https://studio.m-anage.com/agu/agu26/meetingapp.cgi/Paper/2085423"  # any app page works
API_BASE = "https://studio.m-anage.com/agu/agu26/meetingapi.cgi/"
PAPER_PAGE = "https://studio.m-anage.com/agu/agu26/meetingapp.cgi/Paper/"

# @ customize keyword search here (same list/logic as your planner script)
KEYWORDS = [
    "HLS",
    "hls",
    "Harmonized Landsat and Sentinel-2",
    "Harmonized Landsat-Sentinel-2",
    "HLS Low Latency", #?
    "HLS at low latency", #?
    "Harmonized Landsat and Sentinel-2 Low Latency", #?
    "Harmonized Landsat-Sentinel-2 Low Latency", #?
    "HLS VI", #?
    "HLS VIs", #?
    "HLS VI composites", #?
    "HLS Vegetation Indices",
    "HLS Vegetation Index",
    "Harmonized Landsat and Sentinel-2 Vegetation Indices",
    "Harmonized Landsat-Sentinel-2 Vegetation Indices", #?
    "HLS 10 meter", #?
    "HLS 10 m", #?
    "HLS 10m", #?
    "HLS 10-meter", #?
    "Harmonized Landsat and Sentinel-2 10 meter", #?
    "Harmonized Landsat-Sentinel-2 10 meter", #?
    "Harmonized Landsat and Sentinel-2 10 m", #?
    "Harmonized Landsat-Sentinel-2 10 m", #?
    "Harmonized Landsat and Sentinel-2 10m", #?
    "Harmonized Landsat-Sentinel-2 10m", #?
    "Harmonized Landsat and Sentinel-2 10-meter", #?
    "Harmonized Landsat-Sentinel-2 10-meter", #?
    "harmonized landsat",
    "harmonized sentinel",
    "HLS-gpt",
    "HLS-2",
    "Prithvi-EO",
    "Prithvi",
]

HEADLESS = True
CONCURRENCY = 8          # simultaneous API requests inside the browser
CHUNK = 200              # paths per browser call
PROGRAM_ID_RANGE = range(5300, 5460)   # fallback seeds in case Home does not link to every Program
CACHE_FILE = "agu26_program_cache.json"   # delete to force a fresh download
OUT_CSV = "agu26_keyword_title_or_abstract_all_sessions.csv"


def make_keyword_pattern(keywords):
    ordered = sorted(keywords, key=len, reverse=True)
    return re.compile("|".join(r"\b" + re.escape(k) + r"\b" for k in ordered), re.IGNORECASE)


pattern = make_keyword_pattern(KEYWORDS)


def find_matches(text):
    return sorted(set(m.group(0) for m in pattern.finditer(text or "")))


# ============================================================
# Browser-side fetch (same-origin, so no CORS / cookie problems)
# ============================================================
FETCH_JS = """
async ([base, paths, conc]) => {
  const keep = ['Title','Abstract','FinalNumber','FinalPaperNumber','SessionTitle','Session_GoodType',
                'ProgramTitle','ProgramCode','SortableTimeString','TopDisplay','Withdrawn','PresentationLocation',
                'ParentList_ParentEntries'];
  const out = {};
  let i = 0;
  async function worker() {
    while (i < paths.length) {
      const p = paths[i++];
      for (let a = 0; a < 3; a++) {
        try {
          const r = await fetch(base + p, {credentials: 'same-origin'});
          if (r.ok) {
            let j = await r.json();
            if (p.startsWith('Paper/') && j && !Array.isArray(j)) {
              const t = {id: j.id};
              keep.forEach(k => { if (k in j) t[k] = j[k]; });
              j = t;
            }
            out[p] = j; break;
          }
          if (r.status === 404) { out[p] = null; break; }
        } catch (e) {}
        await new Promise(res => setTimeout(res, 600 * (a + 1)));
      }
      if (!(p in out)) out[p] = '__failed__';
    }
  }
  await Promise.all(Array.from({length: conc}, worker));
  return out;
}
"""


async def make_fetcher(page):
    async def fetch_many(paths):
        results = {}
        for s in range(0, len(paths), CHUNK):
            part = paths[s:s + CHUNK]
            results.update(await page.evaluate(FETCH_JS, [API_BASE, part, CONCURRENCY]))
        return results
    return fetch_many


# ============================================================
# Crawl: Home -> (Program ->) Session -> Paper
# ============================================================
FOLLOW = re.compile(r"^(Program|Session|ModuleProgramBook|ModuleSessionsByDay|ListItem|Day)/")
PAPER_REF = re.compile(r"^(?:Paper/|SlotData/Paper)(\d+)")


def refs_in(record):
    """Yield string references stored in a record's ChildList_* fields."""
    if not isinstance(record, dict):
        return
    for k, v in record.items():
        if k.startswith("ChildList_") and isinstance(v, list):
            for x in v:
                if isinstance(x, str):
                    yield x


async def crawl(fetch_many, verbose=True):
    """Return (set of paper ids, dict of visited-record counts by type)."""
    queue = ["Home/0"] + [f"Program/{i}" for i in PROGRAM_ID_RANGE]
    seen, paper_ids, kinds = set(queue), set(), {}
    while queue:
        batch, queue = queue[:CHUNK * 2], queue[CHUNK * 2:]
        res = await fetch_many(batch)
        for path, rec in res.items():
            if not isinstance(rec, dict) or not rec:
                continue
            kinds[path.split("/")[0]] = kinds.get(path.split("/")[0], 0) + 1
            for ref in refs_in(rec):
                m = PAPER_REF.match(ref)
                if m:
                    paper_ids.add(m.group(1))
                elif FOLLOW.match(ref) and ref not in seen:
                    seen.add(ref)
                    queue.append(ref)
        if verbose:
            print(f"  crawled {len(seen) - len(queue)} records | queue {len(queue)} | papers found {len(paper_ids)}")
    return paper_ids, kinds


async def download_papers(fetch_many, paper_ids, cache):
    todo = sorted(p for p in paper_ids if p not in cache)
    print(f"Papers to download: {len(todo)} (cached: {len(paper_ids) - len(todo)})")
    t0 = time.time()
    for s in range(0, len(todo), CHUNK):
        part = todo[s:s + CHUNK]
        res = await fetch_many([f"Paper/{p}" for p in part])
        for path, rec in res.items():
            if isinstance(rec, dict) and rec.get("Title") is not None:
                cache[path.split("/")[1]] = rec
        print(f"  [{min(s + CHUNK, len(todo))}/{len(todo)}] {time.time() - t0:.0f}s")
        json.dump(cache, open(CACHE_FILE, "w"))   # checkpoint
    return cache


async def build_cache():
    cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()
        await page.goto(APP_URL, wait_until="networkidle")
        await asyncio.sleep(3)
        fetch_many = await make_fetcher(page)
        if not cache:
            print("Crawling the program tree ...")
            paper_ids, kinds = await crawl(fetch_many)
            print(f"Records visited by type: {kinds}")
            print(f"Unique papers discovered: {len(paper_ids)}")
            if len(paper_ids) < 1000:
                print("WARNING: far fewer papers than expected (AGU has ~20k+). The crawl did not reach every "
                      "session - send me the 'Records visited' line above.")
            cache = await download_papers(fetch_many, paper_ids, cache)
        await browser.close()
    return cache


# ============================================================
# Matching + CSV
# ============================================================
def clean_html(html):
    soup = BeautifulSoup(html or "", "html.parser")
    return re.sub(r"\s+", " ", soup.get_text(" ", strip=True))


def authors_from_topdisplay(html):
    soup = BeautifulSoup(html or "", "html.parser")
    for s in soup.find_all("sup"):
        s.decompose()
    text = re.sub(r"\s+", " ", soup.get_text(" ", strip=True))
    text = re.split(r"\(1\)", text)[0]          # drop the numbered affiliation list
    return re.sub(r"\s+,", ",", text).strip(" ,")


def paper_to_row(pid, rec):
    title = clean_html(rec.get("Title"))
    abstract = clean_html(rec.get("Abstract"))
    t_hits, a_hits = find_matches(title), find_matches(abstract)
    if not (t_hits or a_hits):
        return None
    st = (rec.get("SortableTimeString") or "").split("_")      # 2026-12-08_17:05:00_17:15:00
    return {
        "presentation_type": rec.get("Session_GoodType", ""),
        "title": title,
        "match_location": "title and abstract" if t_hits and a_hits else "title" if t_hits else "abstract",
        "matched_keywords": sorted(set(t_hits + a_hits)),
        "authors": authors_from_topdisplay(rec.get("TopDisplay")),
        "paper_number": rec.get("FinalPaperNumber") or rec.get("FinalNumber", ""),
        "main_session": clean_html(rec.get("SessionTitle")),
        "program": rec.get("ProgramTitle", ""),
        "date": st[0] if st else "",
        "time": f"{st[1][:5]}-{st[2][:5]}" if len(st) >= 3 else "",
        "presentation_location": rec.get("PresentationLocation", ""),
        "withdrawn": rec.get("Withdrawn", ""),
        "paper_id": pid,
        "url": f"{PAPER_PAGE}{pid}",
        "abstract": abstract,
    }


def make_csv(cache):
    rows = [r for pid, rec in cache.items() if (r := paper_to_row(pid, rec))]
    df = pd.DataFrame(rows)
    if df.empty:
        print("No matches found.")
        return df
    order = {"title": 0, "title and abstract": 1, "abstract": 2}
    df = df.sort_values(["match_location", "date", "time"], key=lambda c: c.map(order) if c.name == "match_location" else c)
    df.to_csv(OUT_CSV, index=False)
    print(f"Papers searched: {len(cache)} | matches: {len(df)} -> {OUT_CSV}")
    print(df["match_location"].value_counts().to_string())
    return df


if __name__ == "__main__":
    cache = asyncio.run(build_cache())
    df = make_csv(cache)
    df[["presentation_type", "title", "match_location", "matched_keywords", "main_session", "date", "time", "url"]] if len(df) else None

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [79.0 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,131 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,141 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,429 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,557 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get: